# 03 — Pretrained Models Sanity Check

This notebook loads three ImageNet-pretrained backbones (ResNet18, ResNet50, DenseNet121) with their final classification layers replaced for 4-class brain tumor classification, then verifies that each model produces the expected output shape on a real batch from the training DataLoader.

## 1. Imports & Setup

Add `src/` to the import path and import the model factory and data loaders.

In [1]:
import sys
from pathlib import Path

# Allow imports from src/
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import torch

from model import MODEL_REGISTRY, get_model
from preprocessing import CLASS_NAMES, IMAGE_SIZE, get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Available models: {list(MODEL_REGISTRY)}")
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

Using device: cuda
Available models: ['resnet18', 'resnet50', 'densenet121']
Classes (4): ['glioma', 'meningioma', 'notumor', 'pituitary']


## 2. Grab a Batch from the Training DataLoader

Pull a single batch so we can run a real forward pass through each model and confirm the output shape is `(batch_size, 4)`.

In [2]:
loaders = get_dataloaders(batch_size=8)
images, labels = next(iter(loaders["train"]))
images = images.to(DEVICE)

print(f"Batch image shape: {tuple(images.shape)}")
print(f"Batch labels:      {labels.tolist()}")

Batch image shape: (8, 3, 224, 224)
Batch labels:      [2, 1, 2, 1, 0, 3, 0, 0]


## 3. Load Each Pretrained Model and Run a Forward Pass

Iterate over the model registry, load each model with `get_model`, run a forward pass on the batch, and assert the output shape is `(batch_size, 4)`.

In [3]:
expected_shape = (images.size(0), len(CLASS_NAMES))
models_loaded = {}

for name in MODEL_REGISTRY:
    model = get_model(name).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(images)

    assert out.shape == expected_shape, (
        f"{name}: expected {expected_shape}, got {tuple(out.shape)}"
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:12s} -> output shape: {tuple(out.shape)}, params: {n_params:,}")
    models_loaded[name] = model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\xethr/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:09<00:00, 4.97MB/s]


resnet18     -> output shape: (8, 4), params: 11,178,564
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\xethr/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:19<00:00, 5.39MB/s]


resnet50     -> output shape: (8, 4), params: 23,516,228
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\xethr/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:05<00:00, 5.50MB/s]


densenet121  -> output shape: (8, 4), params: 6,957,956


## 4. Inspect the Final Classification Layer

Print the replaced final layer for each model to confirm it is a `Linear` layer with `out_features=4`. ResNets use `model.fc`; DenseNet uses `model.classifier`.

In [4]:
for name, model in models_loaded.items():
    head = model.fc if hasattr(model, "fc") else model.classifier
    print(f"{name:12s} final layer: {head}")

resnet18     final layer: Linear(in_features=512, out_features=4, bias=True)
resnet50     final layer: Linear(in_features=2048, out_features=4, bias=True)
densenet121  final layer: Linear(in_features=1024, out_features=4, bias=True)


## 5. Train Each Pretrained Model

Use the generic training loop in `src/train.py` to finetune each of the three pretrained backbones on the brain-tumor splits. The best validation-accuracy checkpoint for each model is saved to `outputs/checkpoints/`.

Import the generic training function from `src/train.py`, build the train/val DataLoaders, and pick a small number of epochs so this cell finishes in a reasonable amount of time.

In [5]:
from train import train_model

loaders = get_dataloaders(batch_size=32)

NUM_EPOCHS = 5
LR = 1e-4

print(f"Train batches: {len(loaders['train'])}  |  Val batches: {len(loaders['val'])}")
print(f"Training each model for {NUM_EPOCHS} epochs at lr={LR}")

Train batches: 140  |  Val batches: 35
Training each model for 5 epochs at lr=0.0001


Loop over the three pretrained backbones, train each one starting from its ImageNet weights, and collect the result dict returned by `train_model` (it holds the best validation accuracy and the path to the saved checkpoint).

In [6]:
results = {}

for name in MODEL_REGISTRY:
    print(f"\n=== {name} ===")
    model = get_model(name)
    results[name] = train_model(
        model,
        loaders,
        num_epochs=NUM_EPOCHS,
        lr=LR,
        checkpoint_name=name,
    )


=== resnet18 ===
Training 'resnet18' for 5 epochs on cuda (lr=0.0001)
  Epoch   1/5 — train_loss: 0.2699 | val_acc: 0.962 | 24.1s
  Epoch   2/5 — train_loss: 0.0645 | val_acc: 0.975 | 24.2s
  Epoch   3/5 — train_loss: 0.0450 | val_acc: 0.981 | 24.2s
  Epoch   4/5 — train_loss: 0.0314 | val_acc: 0.982 | 23.9s
  Epoch   5/5 — train_loss: 0.0166 | val_acc: 0.983 | 24.1s
Best val accuracy: 0.9830  —  saved to C:\Users\xethr\GitHub\badnets-medical-imaging\outputs\checkpoints\best_resnet18.pt

=== resnet50 ===
Training 'resnet50' for 5 epochs on cuda (lr=0.0001)
  Epoch   1/5 — train_loss: 0.3711 | val_acc: 0.963 | 37.2s
  Epoch   2/5 — train_loss: 0.0833 | val_acc: 0.979 | 37.4s
  Epoch   3/5 — train_loss: 0.0408 | val_acc: 0.970 | 38.0s
  Epoch   4/5 — train_loss: 0.0329 | val_acc: 0.984 | 37.4s
  Epoch   5/5 — train_loss: 0.0225 | val_acc: 0.982 | 37.3s
Best val accuracy: 0.9839  —  saved to C:\Users\xethr\GitHub\badnets-medical-imaging\outputs\checkpoints\best_resnet50.pt

=== densenet1

Print a small summary table showing the best validation accuracy each model reached during training.

In [7]:
print(f"{'Model':<14} {'Best Val Acc':<12}")
print("-" * 28)
for name, res in results.items():
    print(f"{name:<14} {res['best_val_acc']:<12.4f}")

Model          Best Val Acc
----------------------------
resnet18       0.9830      
resnet50       0.9839      
densenet121    0.9884      
